# SAM3 포인트 프롬프트 전용 — 기준선 객체 크롭 저장

텍스트 프롬프트 없이 기준선 위에 포인트만 찍어서 SAM3가 세그먼트한 객체를 이미지로 저장한다.

> 포인트/박스/마스크 시각 프롬프트는 `SAM3SemanticPredictor`가 아닌 `SAM` 베이스 클래스를 사용한다. (공식 문서 Visual Prompts 섹션 참고)

In [2]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import SAM

In [3]:
VIDEO_PATH   = "videos/test.mp4"
OUT_DIR      = Path("sam3_crops")
OUT_DIR.mkdir(exist_ok=True)

LINE_Y_RATIO = 0.85   # 기준선 위치 (프레임 높이 비율)
POINT_STEP   = 60     # 포인트 간격 (픽셀)

## Step 1 — SAM 모델 로드

In [4]:
# 포인트/박스 시각 프롬프트는 SAM 베이스 클래스 사용 (SAM3SemanticPredictor 아님)
model = SAM("sam3.pt")

## Step 2 — 프레임 로드 & 기준선 미리보기

In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

fh, fw = frames[0].shape[:2]
line_y   = int(fh * LINE_Y_RATIO)
points_x = list(range(0, fw, POINT_STEP))

print(f"총 {len(frames)} 프레임  ({fw}×{fh})")
print(f"기준선 y={line_y}px  포인트 {len(points_x)}개")

preview = frames[0].copy()
cv2.line(preview, (0, line_y), (fw - 1, line_y), (0, 255, 255), 3)
for px in points_x:
    cv2.circle(preview, (px, line_y), 6, (0, 80, 255), -1)
cv2.imwrite("sam3_point_preview.jpg", preview)
print("미리보기 저장: sam3_point_preview.jpg")

총 649 프레임  (1080×1920)
기준선 y=1632px  포인트 18개
미리보기 저장: sam3_point_preview.jpg


## Step 3 — 포인트 프롬프트로 세그먼트 & 크롭 저장

기준선 위 포인트를 foreground(`label=1`)로 넘긴다.  
마스크별로 bbox 크롭을 `sam3_crops/` 에 저장한다.

In [6]:
# 기준선 위 포인트 배열: [[x, y], ...]  label 1 = foreground
prompt_points = [[px, line_y] for px in points_x]
prompt_labels = [1] * len(prompt_points)

saved_total = 0

for frame_idx, frame in enumerate(frames):
    results = model.predict(
        source=frame,
        points=[prompt_points],
        labels=[prompt_labels],
        verbose=False,
    )

    r = results[0]
    if r.masks is None:
        continue

    saved_this = 0
    for mask_idx, mask in enumerate(r.masks.data.cpu().numpy()):
        mask_r = cv2.resize(mask, (fw, fh), interpolation=cv2.INTER_NEAREST)

        ys, xs = np.where(mask_r > 0)
        if len(xs) == 0:
            continue

        x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()

        crop = frame[y1:y2, x1:x2].copy()
        crop[mask_r[y1:y2, x1:x2] == 0] = 0   # 마스크 밖 배경 검정

        cv2.imwrite(str(OUT_DIR / f"f{frame_idx:05d}_m{mask_idx:02d}.jpg"), crop)
        saved_this += 1

    saved_total += saved_this
    if frame_idx % 30 == 0:
        print(f"  frame {frame_idx:4d}  마스크={saved_this}  누적={saved_total}")

print(f"\n완료: 총 {saved_total}개 크롭 저장 → {OUT_DIR}/")

WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'resize'
> Overload resolution failed:
>  - src data type = bool is not supported
>  - Expected Ptr<cv::UMat> for argument 'src'


## Step 4 — 결과 확인 (첫 프레임 타일 이미지)

In [ ]:
first_crops = sorted(OUT_DIR.glob("f00000_*.jpg"))
if not first_crops:
    print("첫 프레임 크롭 없음 — LINE_Y_RATIO 또는 POINT_STEP 조정 필요")
else:
    th = 200
    tiles = []
    for p in first_crops:
        img = cv2.imread(str(p))
        h, w = img.shape[:2]
        tiles.append(cv2.resize(img, (max(1, int(w * th / h)), th)))
    cv2.imwrite("sam3_point_check.jpg", np.hstack(tiles))
    print(f"첫 프레임 크롭 {len(tiles)}개 → sam3_point_check.jpg")